# 1. IMPORTS & CHARGEMENT

In [1]:
import sys
import os
sys.path.append(os.path.abspath("../"))

In [2]:
import pandas as pd

from src.clustering import perform_clustering
from src.profiles import build_cluster_profiles
from src.recommendation import get_user_vector, recommend_clusters

df = pd.read_csv("../data/pos_cotonou.csv")
df.head()

,city,lat,lon,category,name,source
0,Cotonou,6.347745,2.403413,hotel,Arcada Marina,OSM
1,Cotonou,6.349609,2.386550,atm,Ecobank,OSM
2,Cotonou,6.356871,2.404173,hotel,Chant d'oiseau,OSM
3,Cotonou,6.358407,2.431501,pharmacy,Pharmacie Jonquet,OSM
4,Cotonou,6.359506,2.419645,restaurant,Costa Rica,OSM


# 2. CLUSTERING

In [3]:
df, kmeans = perform_clustering(df, n_clusters=9)

df.head()

C:\Users\Bernard\anaconda3\envs\tensorflow\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


,city,lat,lon,category,name,source,cluster
0,Cotonou,6.347745,2.403413,hotel,Arcada Marina,OSM,4
1,Cotonou,6.349609,2.386550,atm,Ecobank,OSM,4
2,Cotonou,6.356871,2.404173,hotel,Chant d'oiseau,OSM,4
3,Cotonou,6.358407,2.431501,pharmacy,Pharmacie Jonquet,OSM,5
4,Cotonou,6.359506,2.419645,restaurant,Costa Rica,OSM,1


# 3. CONSTRUCTION DES PROFILS DE CLUSTERS

### Chaque cluster est représenté par un vecteur décrivant le nombre de points de service par catégorie.

In [4]:
cluster_profiles,cluster_profiles_norm = build_cluster_profiles(df)

cluster_profiles

category,atm,driving_school,hospital,hotel,music_school,pharmacy,restaurant,school
cluster,,,,,,,,
0,1,0,1,3,0,5,2,10
1,4,2,5,12,0,14,27,15
2,1,2,3,8,0,8,22,22
3,0,0,3,6,0,11,10,34
4,5,0,2,21,1,4,37,16
5,2,2,3,23,0,16,58,12
6,0,4,3,4,0,11,9,14
7,1,0,3,8,0,9,15,21
8,0,0,0,5,0,3,12,9


# 4. NORMALISATION
## La normalisation permet de comparer les clusters indépendamment de leur taille.

In [5]:
cluster_profiles_norm

category,atm,driving_school,hospital,hotel,music_school,pharmacy,restaurant,school
cluster,,,,,,,,
0,0.084515,0.000000,0.084515,0.253546,0.00000,0.422577,0.169031,0.845154
1,0.109312,0.054656,0.136641,0.327937,0.00000,0.382594,0.737859,0.409922
2,0.030015,0.060030,0.090045,0.240120,0.00000,0.240120,0.660330,0.660330
3,0.000000,0.000000,0.079556,0.159111,0.00000,0.291704,0.265186,0.901632
4,0.108799,0.000000,0.043519,0.456954,0.02176,0.087039,0.805109,0.348155
5,0.030464,0.030464,0.045696,0.350340,0.00000,0.243714,0.883465,0.182786
6,0.000000,0.190910,0.143182,0.190910,0.00000,0.525001,0.429547,0.668184
7,0.034900,0.000000,0.104701,0.279202,0.00000,0.314102,0.523504,0.732905
8,0.000000,0.000000,0.000000,0.310685,0.00000,0.186411,0.745644,0.559233


#  5. PRÉFÉRENCES UTILISATEUR

In [16]:
user_preferences = {
    "restaurant": 1,
    "school": 10,
    "hospital": 2,
    "hotel": 20,
    "pharmacy": 0,
    "atm": 1
}

user_preferences

{'restaurant': 1,
 'school': 10,
 'hospital': 2,
 'hotel': 20,
 'pharmacy': 0,
 'atm': 1}

# 6. CONSTRUCTION DU VECTEUR UTILISATEUR
## On transforme les préférences en vecteur compatible avec les clusters.

In [17]:
user_vector = get_user_vector(user_preferences, cluster_profiles.columns)

user_vector

array([ 1,  0,  2, 20,  0,  0,  1, 10])

# 7. CALCUL DE SIMILARITÉ (COSINUS)
## On utilise la similarité cosinus pour mesurer la proximité entre le besoin utilisateur et chaque cluster.

$$
\text{sim}(A,B) = \frac{A \cdot B}{||A|| \times ||B||}
$$

## Les 4 clusters les plus similaires sont sélectionnés.

In [19]:
top_clusters, similarities = recommend_clusters(
    user_vector,
    cluster_profiles_norm
)

top_clusters

array([0, 7, 4, 3], dtype=int64)

In [20]:
# from scipy.spatial.distance import cdist

# distances = cdist(
#     [user_vector],
#     cluster_profiles,
#     metric='euclidean'
# )[0]

# top_clusters = distances.argsort()[:4]
# top_clusters

# 8. VISUALISATION DES CLUSTERS RECOMMANDÉS

In [21]:
from src.visualization import plot_cluster_map

for cluster_id in top_clusters:
    plot_cluster_map(df, cluster_id)

Carte cluster 0 sauvegardée : ../data/maps/cluster_0.html
Carte cluster 7 sauvegardée : ../data/maps/cluster_7.html
Carte cluster 4 sauvegardée : ../data/maps/cluster_4.html
Carte cluster 3 sauvegardée : ../data/maps/cluster_3.html
